<a href="https://colab.research.google.com/github/anshuman-dataverse/transformer-recipes-imdb/blob/main/transformer_recipes_imdb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Transformers for Recipe Generation and Sentiment Classification

**Part I** adapts the GPT example to generate recipes from the Epicurious dataset and compares it with
the autoregressive LSTM. **Part II** turns the same GPT block into a binary classifier for IMDB reviews
and compares it with an LSTM classifier. No pretrained weights are used anywhere.

In [ ]:
%matplotlib inline
import os
import re
import json
import time
import string
import textwrap
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, losses, callbacks
from tensorflow.keras.datasets import imdb

VOCAB_SIZE = 10000
MAX_LEN = 200
EMBEDDING_DIM = 100
N_UNITS = 128
VALIDATION_SPLIT = 0.2
SEED = 42
LOAD_MODEL = False
BATCH_SIZE = 32
EPOCHS = 20

# transformer-specific: N_UNITS sets the attention key dimension, the feed-forward layer is 2x wider
N_HEADS = 2
KEY_DIM = N_UNITS
FEED_FORWARD_DIM = 2 * N_UNITS

DATA_PATH = 'data/epirecipes/full_format_recipes.json'
if not os.path.exists(DATA_PATH):
    # public Kaggle dataset, downloaded without credentials
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'kagglehub'], check=True)
    import kagglehub
    kagglehub.dataset_download('hugodarwood/epirecipes', output_dir='data/epirecipes')
os.makedirs('models', exist_ok=True)
keras.utils.set_random_seed(SEED)
print('tensorflow', tf.__version__, '| keras', keras.__version__, '| GPUs:', tf.config.list_physical_devices('GPU'))

## Shared building blocks

The GPT block from the example: token + position embedding, one multi-head self-attention layer with a
causal mask, a position-wise feed-forward network, and residual connections with layer normalisation.
The only change from the example is a `causal` flag, so Part II can also try attention without the mask.

In [ ]:
class TokenAndPositionEmbedding(layers.Layer):
    def __init__(self, max_len, vocab_size, embed_dim, **kwargs):
        super().__init__(**kwargs)
        self.token_emb = layers.Embedding(vocab_size, embed_dim)
        self.pos_emb = layers.Embedding(max_len, embed_dim)

    def call(self, x):
        positions = keras.ops.arange(keras.ops.shape(x)[-1])
        return self.token_emb(x) + self.pos_emb(positions)


class TransformerBlock(layers.Layer):
    def __init__(self, num_heads, key_dim, embed_dim, ff_dim, dropout_rate=0.1, causal=True, **kwargs):
        super().__init__(**kwargs)
        self.causal = causal
        self.attn = layers.MultiHeadAttention(num_heads, key_dim, output_shape=embed_dim)
        self.dropout_1 = layers.Dropout(dropout_rate)
        self.ln_1 = layers.LayerNormalization(epsilon=1e-6)
        self.ffn_1 = layers.Dense(ff_dim, activation='relu')
        self.ffn_2 = layers.Dense(embed_dim)
        self.dropout_2 = layers.Dropout(dropout_rate)
        self.ln_2 = layers.LayerNormalization(epsilon=1e-6)

    def call(self, inputs, training=False):
        attn = self.attn(inputs, inputs, use_causal_mask=self.causal, training=training)
        out_1 = self.ln_1(inputs + self.dropout_1(attn, training=training))
        ffn = self.ffn_2(self.ffn_1(out_1))
        return self.ln_2(out_1 + self.dropout_2(ffn, training=training))


def plot_history(histories, title):
    fig, ax = plt.subplots(figsize=(7, 3.6))
    for name, h in histories.items():
        line, = ax.plot(range(1, len(h['loss']) + 1), h['loss'], label=f'{name} train')
        ax.plot(range(1, len(h['val_loss']) + 1), h['val_loss'], '--', color=line.get_color(), label=f'{name} val')
    ax.set_xlabel('epoch')
    ax.set_ylabel('loss')
    ax.set_title(title)
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()


class BestWeights(callbacks.Callback):
    # keeps the weights of the epoch with the lowest validation loss, without stopping training
    def on_train_begin(self, logs=None):
        self.best, self.best_epoch, self.weights = np.inf, None, None

    def on_epoch_end(self, epoch, logs=None):
        if logs['val_loss'] < self.best:
            self.best, self.best_epoch = logs['val_loss'], epoch + 1
            self.weights = self.model.get_weights()


class EpochTimer(callbacks.Callback):
    def on_train_begin(self, logs=None):
        self.times = []

    def on_epoch_begin(self, epoch, logs=None):
        self._t = time.time()

    def on_epoch_end(self, epoch, logs=None):
        self.times.append(time.time() - self._t)


def fit_or_load(model, path, train_ds, val_ds, extra_callbacks=()):
    timer = EpochTimer()
    if LOAD_MODEL and os.path.exists(path):
        model.load_weights(path)
        return None, timer
    hist = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS,
                     callbacks=[timer, *extra_callbacks], verbose=2)
    model.save_weights(path)
    return hist.history, timer

# Part I — Recipe generation

## 1. Data

Each recipe becomes one string, `Recipe for <title> | <directions>`, the same format the LSTM example
uses. Punctuation is split into separate tokens so the model can learn sentence boundaries and list
structure. Recipes are shuffled with the fixed seed and split 80/20 into train and validation.

In [ ]:
with open(DATA_PATH) as f:
    recipe_data = json.load(f)

filtered = [r for r in recipe_data if r.get('title') and r.get('directions')]
recipes = ['Recipe for ' + r['title'] + ' | ' + ' '.join(r['directions']) for r in filtered]


def pad_punctuation(s):
    s = re.sub(f'([{re.escape(string.punctuation)}])', r' \1 ', s)
    return re.sub(' +', ' ', s).strip()


recipes = [pad_punctuation(r) for r in recipes]
rng = np.random.default_rng(SEED)
order = rng.permutation(len(recipes))
n_val = int(len(recipes) * VALIDATION_SPLIT)
val_text = [recipes[i] for i in order[:n_val]]
train_text = [recipes[i] for i in order[n_val:]]

lengths = np.array([len(r.split()) for r in recipes])
print(f'{len(recipe_data)} raw recipes -> {len(recipes)} with a title and directions')
print(f'train {len(train_text)} | validation {len(val_text)}')
print(f'tokens per recipe: median {np.median(lengths):.0f}, '
      f'{(lengths > MAX_LEN).mean():.1%} longer than MAX_LEN={MAX_LEN}')
print()
print(textwrap.fill(train_text[0], 110))

In [ ]:
vectorize_layer = layers.TextVectorization(standardize='lower', max_tokens=VOCAB_SIZE,
                                           output_mode='int', output_sequence_length=MAX_LEN + 1)
vectorize_layer.adapt(tf.data.Dataset.from_tensor_slices(train_text).batch(256))
vocab = vectorize_layer.get_vocabulary()
word_to_index = {w: i for i, w in enumerate(vocab)}


def to_ids(texts):
    return np.concatenate([vectorize_layer(tf.constant(texts[i:i + 1024])).numpy()
                           for i in range(0, len(texts), 1024)])


train_ids, val_ids = to_ids(train_text), to_ids(val_text)


def lm_dataset(ids, shuffle):
    ds = tf.data.Dataset.from_tensor_slices((ids[:, :-1], ids[:, 1:]))
    if shuffle:
        ds = ds.shuffle(len(ids), seed=SEED)
    return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)


lm_train_ds, lm_val_ds = lm_dataset(train_ids, True), lm_dataset(val_ids, False)
print('vocabulary', len(vocab), '| first tokens:', vocab[:12])
print('unknown-token rate in validation:',
      f'{(val_ids == 1).sum() / (val_ids > 0).sum():.2%}')

## 2. Models

**GPT** — the example architecture with the recipe vocabulary: embedding → one causal transformer block →
softmax over 10 000 tokens at every position. **LSTM** — the autoregressive LSTM example: embedding →
LSTM(128) → softmax at every position. Both use the same tokens, loss and optimiser, so differences in
the samples come from the architecture alone.

In [ ]:
def build_gpt():
    inputs = layers.Input(shape=(None,), dtype='int32')
    x = TokenAndPositionEmbedding(MAX_LEN, VOCAB_SIZE, EMBEDDING_DIM)(inputs)
    x = TransformerBlock(N_HEADS, KEY_DIM, EMBEDDING_DIM, FEED_FORWARD_DIM)(x)
    outputs = layers.Dense(VOCAB_SIZE, activation='softmax')(x)
    model = keras.Model(inputs, outputs, name='gpt_recipes')
    model.compile('adam', loss=losses.SparseCategoricalCrossentropy())
    return model


def build_lstm_lm():
    inputs = layers.Input(shape=(None,), dtype='int32')
    x = layers.Embedding(VOCAB_SIZE, EMBEDDING_DIM)(inputs)
    x = layers.LSTM(N_UNITS, return_sequences=True)(x)
    outputs = layers.Dense(VOCAB_SIZE, activation='softmax')(x)
    model = keras.Model(inputs, outputs, name='lstm_recipes')
    model.compile('adam', loss=losses.SparseCategoricalCrossentropy())
    return model


keras.utils.set_random_seed(SEED)
gpt = build_gpt()
lstm_lm = build_lstm_lm()
gpt.summary()
lstm_lm.summary()

## 3. Training

Both models train for 20 epochs with batch size 32. The Keras loss also counts the padding positions
after each recipe ends. Those positions are easy to predict, so the loss below the plot is recomputed on
real tokens only and reported as perplexity.

In [ ]:
keras.utils.set_random_seed(SEED)
gpt_hist, gpt_timer = fit_or_load(gpt, 'models/gpt_recipes.weights.h5', lm_train_ds, lm_val_ds)

In [ ]:
keras.utils.set_random_seed(SEED)
lstm_lm_hist, lstm_lm_timer = fit_or_load(lstm_lm, 'models/lstm_recipes.weights.h5', lm_train_ds, lm_val_ds)

In [ ]:
if gpt_hist and lstm_lm_hist:
    plot_history({'GPT': gpt_hist, 'LSTM': lstm_lm_hist}, 'Recipe language models')


def masked_perplexity(model, ds):
    total, count, correct = 0.0, 0, 0
    for x, y in ds:
        p = model(x, training=False)
        mask = keras.ops.cast(y > 0, 'float32')
        ce = losses.sparse_categorical_crossentropy(y, p)
        total += float(keras.ops.sum(ce * mask))
        count += float(keras.ops.sum(mask))
        correct += float(keras.ops.sum(keras.ops.cast(keras.ops.equal(keras.ops.cast(keras.ops.argmax(p, -1), 'int32'), keras.ops.cast(y, 'int32')), 'float32') * mask))
    return np.exp(total / count), correct / count


lm_rows = []
for name, model, hist, timer in [('GPT', gpt, gpt_hist, gpt_timer), ('LSTM', lstm_lm, lstm_lm_hist, lstm_lm_timer)]:
    ppl, acc = masked_perplexity(model, lm_val_ds)
    lm_rows.append({'model': name, 'params': model.count_params(),
                    'final train loss': hist['loss'][-1], 'final val loss': hist['val_loss'][-1],
                    'best val epoch': int(np.argmin(hist['val_loss'])) + 1,
                    'val perplexity (real tokens)': ppl, 'val next-token acc': acc,
                    'sec / epoch': np.mean(timer.times)})
lm_df = pd.DataFrame(lm_rows).set_index('model')
lm_df.round(3)

## 4. Generation and temperature

Generation follows the example: start from a prompt, predict the next-token distribution, rescale it with
a temperature $T$, sample, append, repeat until the model emits padding (end of recipe) or MAX_LEN is reached.

$$p_i^{(T)} = \frac{p_i^{1/T}}{\sum_j p_j^{1/T}}$$

Low $T$ sharpens the distribution towards the most likely token; high $T$ flattens it. Sequences are
generated in batches on a fixed-length input so that each step is one compiled forward pass.

In [ ]:
gen_rng = np.random.default_rng(SEED)


def make_steps(model):
    # the full model scores every position; the next-token step only needs the head at position t - 1
    body = keras.Model(model.input, model.layers[-2].output)
    head = model.layers[-1]

    @tf.function(reduce_retracing=True)
    def full(x):
        return model(x, training=False)

    @tf.function(reduce_retracing=True)
    def next_token(x, t):
        return head(body(x, training=False)[:, t - 1, :])
    return full, next_token


steps, next_steps = {}, {}
for _name, _model in [('GPT', gpt), ('LSTM', lstm_lm)]:
    steps[_name], next_steps[_name] = make_steps(_model)


def generate(model_name, prompt, n, temperature, max_tokens=MAX_LEN):
    prompt_ids = [word_to_index.get(w, 1) for w in pad_punctuation(prompt).lower().split()]
    seqs = np.zeros((n, max_tokens), dtype='int32')
    seqs[:, :len(prompt_ids)] = prompt_ids
    ended = np.zeros(n, dtype=bool)
    for t in range(len(prompt_ids), max_tokens):
        probs = next_steps[model_name](tf.constant(seqs), tf.constant(t)).numpy().astype('float64')
        logits = np.log(probs + 1e-12) / temperature
        nxt = np.argmax(logits + gen_rng.gumbel(size=logits.shape), axis=-1)  # Gumbel-max = sampling
        nxt[ended] = 0
        seqs[:, t] = nxt
        ended |= nxt == 0
        if ended.all():
            break
    out = []
    for s in seqs:
        s = list(s)
        out.append(s[:s.index(0)] if 0 in s else s)
    return out


def detok(ids):
    text = ' '.join(vocab[i] if i < len(vocab) else '[UNK]' for i in ids)
    return re.sub(r'\s+([.,;:!?)%])', r'\1', text).replace('( ', '(')


def show(model_name, prompt, temperature, n=1, width=110):
    for ids in generate(model_name, prompt, n, temperature):
        print(textwrap.fill(detok(ids), width))
        print()

In [ ]:
TEMPERATURES = [0.2, 0.5, 1.0, 1.5]
PROMPTS = ['recipe for roasted vegetables |', 'recipe for chocolate', 'recipe for']

for prompt in PROMPTS:
    for model_name in ['GPT', 'LSTM']:
        print('=' * 110)
        print(f'{model_name}  |  prompt: "{prompt}"')
        print('=' * 110)
        for T in TEMPERATURES:
            print(f'--- T = {T}')
            show(model_name, prompt, T)

### Next-token distribution at different temperatures

The effect of $T$ is easiest to see on one prediction. Below is the GPT's distribution for the token after
`recipe for roasted vegetables | preheat oven to`, rescaled at each temperature.

In [ ]:
probe = 'recipe for roasted vegetables | preheat oven to'
probe_ids = [word_to_index.get(w, 1) for w in probe.split()]
p = gpt(np.array([probe_ids]), training=False).numpy()[0, -1].astype('float64')

fig, axes = plt.subplots(1, len(TEMPERATURES), figsize=(14, 3), sharey=True)
for ax, T in zip(axes, TEMPERATURES):
    q = p ** (1 / T)
    q /= q.sum()
    top = np.argsort(q)[::-1][:8]
    ax.barh([vocab[i] if i < len(vocab) else '[UNK]' for i in top][::-1], q[top][::-1], color='tab:blue')
    ax.set_title(f'T = {T}  (top-1 = {q[top[0]]:.2f})', fontsize=10)
    ax.set_xlim(0, 1)
plt.suptitle(f'GPT next-token probabilities after "{probe}"', fontsize=11)
plt.tight_layout()
plt.show()

### Quantitative comparison

For each model and temperature, 50 recipes are sampled from the open prompt `recipe for`, so the model
also invents the title. The metrics are:

| metric | what it measures |
|---|---|
| **length** | mean tokens generated before the model ends the recipe |
| **ended** | share of samples where the model ended the recipe itself rather than hitting MAX_LEN |
| **repeated 4-grams** | share of 4-grams within a sample that repeat an earlier 4-gram in the same sample (looping) |
| **distinct-1 / distinct-2** | unique unigrams / bigrams ÷ total, pooled over the 50 samples (diversity) |
| **novel 4-grams** | share of generated 4-grams that never occur in the training recipes (creativity vs copying) |
| **novel titles** | share of generated titles that are not titles in the training set |
| **self-PPL / cross-PPL** | perplexity of the text under the model that wrote it / under the other model (fluency, judged by a model that did not write it) |

In [ ]:
N_SAMPLES = 50
EVAL_TEMPERATURES = [0.2, 0.5, 0.75, 1.0, 1.25, 1.5]


def ngrams(seq, n):
    return [tuple(seq[i:i + n]) for i in range(len(seq) - n + 1)]


train_4grams = set()
for row in train_ids:
    seq = row[row > 0].tolist()
    train_4grams.update(ngrams(seq, 4))
bar = word_to_index['|']
train_titles = {tuple(r[:r.index(bar)]) for r in (list(x[x > 0]) for x in train_ids) if bar in r}


def text_perplexity(model_name, samples):
    total, count = 0.0, 0
    for s in samples:
        s = s[:MAX_LEN]
        x = np.zeros((1, MAX_LEN), dtype='int32')
        x[0, :len(s)] = s
        p = steps[model_name](tf.constant(x)).numpy()[0]
        idx = np.arange(2, len(s))  # skip the fixed 'recipe for' prompt
        total -= np.log(p[idx - 1, np.array(s)[idx]] + 1e-12).sum()
        count += len(idx)
    return np.exp(total / count)


def sample_metrics(model_name, samples):
    all_1 = [g for s in samples for g in ngrams(s, 1)]
    all_2 = [g for s in samples for g in ngrams(s, 2)]
    all_4 = [g for s in samples for g in ngrams(s, 4)]
    rep = [1 - len(set(ngrams(s, 4))) / max(len(ngrams(s, 4)), 1) for s in samples]
    titles = [tuple(s[:s.index(bar)]) for s in samples if bar in s]
    other = 'LSTM' if model_name == 'GPT' else 'GPT'
    return {'length': np.mean([len(s) for s in samples]),
            'ended': np.mean([len(s) < MAX_LEN for s in samples]),
            'repeated 4-grams': np.mean(rep),
            'distinct-1': len(set(all_1)) / len(all_1),
            'distinct-2': len(set(all_2)) / len(all_2),
            'novel 4-grams': np.mean([g not in train_4grams for g in all_4]),
            'novel titles': np.mean([t not in train_titles for t in titles]) if titles else np.nan,
            'self-PPL': text_perplexity(model_name, samples),
            'cross-PPL': text_perplexity(other, samples)}


rows, sample_bank = [], {}
for model_name in ['GPT', 'LSTM']:
    for T in EVAL_TEMPERATURES:
        samples = generate(model_name, 'recipe for', N_SAMPLES, T)
        sample_bank[(model_name, T)] = samples
        rows.append({'model': model_name, 'T': T, **sample_metrics(model_name, samples)})

# reference values from held-out human-written recipes
human = [list(r[r > 0]) for r in val_ids[:N_SAMPLES]]
ref = sample_metrics('GPT', human)
ref['self-PPL'], ref['cross-PPL'] = text_perplexity('GPT', human), text_perplexity('LSTM', human)
rows.append({'model': 'human (val)', 'T': np.nan, **ref})
gen_df = pd.DataFrame(rows)
gen_df.rename(columns={'self-PPL': 'self-PPL (GPT for human)', 'cross-PPL': 'cross-PPL (LSTM for human)'}).round(3)

In [ ]:
metrics = [('repeated 4-grams', 'repetition within a sample'), ('distinct-2', 'diversity across samples'),
           ('novel 4-grams', 'novelty vs training data'), ('cross-PPL', 'perplexity under the other model (log)')]
fig, axes = plt.subplots(1, 4, figsize=(16, 3.4))
for ax, (m, title) in zip(axes, metrics):
    for name, style in [('GPT', 'o-'), ('LSTM', 's-')]:
        d = gen_df[gen_df.model == name]
        ax.plot(d['T'], d[m], style, label=name)
    ax.axhline(gen_df.loc[gen_df.model == 'human (val)', m].item(), color='gray', ls=':', label='human')
    ax.set_title(f'{m}\n{title}', fontsize=10)
    ax.set_xlabel('temperature')
    ax.grid(alpha=0.3)
    if m == 'cross-PPL':
        ax.set_yscale('log')
axes[0].legend(fontsize=8)
plt.tight_layout()
plt.show()

# Part II — Sentiment classification

## 1. Data

IMDB reviews are already integer-encoded. With `num_words=10000`, index 0 is padding, 1 marks the start,
2 is out-of-vocabulary and real words start at 3. Reviews are padded or truncated to 200 tokens.
Padding goes at the front and truncation keeps the **last** 200 tokens, so the closing verdict of a long
review is kept and both models see real tokens at the final positions. 20% of the 25 000 training reviews
are held out for validation. The 25 000 test reviews are only used for the final comparison.

In [ ]:
(x_train_full, y_train_full), (x_test_raw, y_test) = imdb.load_data(num_words=VOCAB_SIZE)
raw_test_len = np.array([len(s) for s in x_test_raw])

x_train_full = keras.utils.pad_sequences(x_train_full, maxlen=MAX_LEN)
x_test = keras.utils.pad_sequences(x_test_raw, maxlen=MAX_LEN)

perm = np.random.default_rng(SEED).permutation(len(x_train_full))
n_val = int(len(perm) * VALIDATION_SPLIT)
x_val, y_val = x_train_full[perm[:n_val]], y_train_full[perm[:n_val]]
x_train, y_train = x_train_full[perm[n_val:]], y_train_full[perm[n_val:]]

print(f'train {x_train.shape} | val {x_val.shape} | test {x_test.shape}')
print(f'positive share: train {y_train.mean():.3f}, val {y_val.mean():.3f}, test {y_test.mean():.3f}')
print(f'test reviews longer than {MAX_LEN} tokens: {(raw_test_len > MAX_LEN).mean():.1%} '
      f'(median length {np.median(raw_test_len):.0f})')


def clf_dataset(x, y, shuffle):
    ds = tf.data.Dataset.from_tensor_slices((x, y))
    if shuffle:
        ds = ds.shuffle(len(x), seed=SEED)
    return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)


clf_train_ds = clf_dataset(x_train, y_train, True)
clf_val_ds = clf_dataset(x_val, y_val, False)

## 2. Models

**GPT classifier** — the Part I model with the language-model head replaced. The 10 000-way softmax that
predicts the next token is removed; the block's output vectors are averaged over the 200 positions and fed
to a sigmoid unit. Everything below the head (embeddings, causal attention, feed-forward) is unchanged.

**Bidirectional-attention variant** — the same model with the causal mask switched off, so every position
attends to the whole review. This is not required by the assignment but isolates how much the GPT's
left-to-right mask costs when the task is to read a whole review rather than predict the next word.

**LSTM** — embedding → LSTM(128) → sigmoid, using the final hidden state.

In [ ]:
def build_gpt_classifier(causal=True):
    inputs = layers.Input(shape=(MAX_LEN,), dtype='int32')
    x = TokenAndPositionEmbedding(MAX_LEN, VOCAB_SIZE, EMBEDDING_DIM)(inputs)
    x = TransformerBlock(N_HEADS, KEY_DIM, EMBEDDING_DIM, FEED_FORWARD_DIM, causal=causal)(x)
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dropout(0.1)(x)
    outputs = layers.Dense(1, activation='sigmoid')(x)
    model = keras.Model(inputs, outputs, name='gpt_classifier' + ('' if causal else '_bidirectional'))
    model.compile('adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model


def build_lstm_classifier():
    inputs = layers.Input(shape=(MAX_LEN,), dtype='int32')
    x = layers.Embedding(VOCAB_SIZE, EMBEDDING_DIM)(inputs)
    x = layers.LSTM(N_UNITS)(x)
    outputs = layers.Dense(1, activation='sigmoid')(x)
    model = keras.Model(inputs, outputs, name='lstm_classifier')
    model.compile('adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model


keras.utils.set_random_seed(SEED)
gpt_clf = build_gpt_classifier()
gpt_clf.summary()

## 3. Training

Each model trains for the full 20 epochs. A callback records the epoch with the lowest validation loss so
the test set can be scored both at the last epoch and at the best-validation epoch.

In [ ]:
clf_models, clf_hist, clf_timers, clf_best = {}, {}, {}, {}
for name, builder in [('GPT (causal)', lambda: build_gpt_classifier(True)),
                      ('GPT (bidirectional)', lambda: build_gpt_classifier(False)),
                      ('LSTM', build_lstm_classifier)]:
    print(f'\n### {name}')
    keras.utils.set_random_seed(SEED)
    model = builder()
    best = BestWeights()
    path = 'models/' + re.sub(r'\W+', '_', name.lower()).strip('_') + '.weights.h5'
    hist, timer = fit_or_load(model, path, clf_train_ds, clf_val_ds, [best])
    clf_models[name], clf_hist[name], clf_timers[name], clf_best[name] = model, hist, timer, best

In [ ]:
plot_history({k: v for k, v in clf_hist.items() if v}, 'IMDB — training and validation loss')

fig, ax = plt.subplots(figsize=(7, 3.4))
for name, h in clf_hist.items():
    line, = ax.plot(range(1, EPOCHS + 1), h['accuracy'], label=f'{name} train')
    ax.plot(range(1, EPOCHS + 1), h['val_accuracy'], '--', color=line.get_color(), label=f'{name} val')
ax.set_xlabel('epoch')
ax.set_ylabel('accuracy')
ax.set_title('IMDB — training and validation accuracy')
ax.legend(fontsize=8)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

loss_table = pd.DataFrame({f'{name} {split}': h[key] for name, h in clf_hist.items()
                           for split, key in [('train', 'loss'), ('val', 'val_loss')]},
                          index=pd.RangeIndex(1, EPOCHS + 1, name='epoch'))
loss_table.round(4)

## 4. Test accuracy

In [ ]:
rows, test_probs = [], {}
for name, model in clf_models.items():
    h, best = clf_hist[name], clf_best[name]
    final_loss, final_acc = model.evaluate(x_test, y_test, batch_size=256, verbose=0)
    final_weights = model.get_weights()
    model.set_weights(best.weights)
    best_loss, best_acc = model.evaluate(x_test, y_test, batch_size=256, verbose=0)
    test_probs[name] = model.predict(x_test, batch_size=256, verbose=0).ravel()
    model.set_weights(final_weights)
    rows.append({'model': name, 'params': model.count_params(),
                 'final train loss': h['loss'][-1], 'final val loss': h['val_loss'][-1],
                 'min val loss': min(h['val_loss']), 'best epoch': best.best_epoch,
                 'test acc (epoch 20)': final_acc, 'test acc (best epoch)': best_acc,
                 'test loss (epoch 20)': final_loss, 'sec / epoch': np.mean(clf_timers[name].times)})
clf_df = pd.DataFrame(rows).set_index('model')
clf_df.round(4)

### Accuracy by review length

Reviews longer than 200 tokens are truncated, and long reviews are also where long-range context should
matter most. Test accuracy (best-epoch weights) is broken down by the original review length.

In [ ]:
bins = [0, 100, 200, 400, np.inf]
labels = ['<100', '100-200', '200-400', '>400']
bucket = pd.cut(raw_test_len, bins=bins, labels=labels)
by_len = pd.DataFrame({name: pd.Series((p > 0.5) == y_test).groupby(bucket, observed=True).mean()
                       for name, p in test_probs.items()})
by_len['n reviews'] = pd.Series(bucket).value_counts().reindex(labels).values
by_len.round(4)

In [ ]:
ax = by_len.drop(columns='n reviews').plot.bar(figsize=(7, 3.4), rot=0, width=0.8)
ax.set_ylim(0.75, 0.95)
ax.set_xlabel('original review length (tokens)')
ax.set_ylabel('test accuracy')
ax.set_title('Test accuracy by review length (best-epoch weights)')
ax.grid(alpha=0.3, axis='y')
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

agree = np.mean((test_probs['GPT (causal)'] > 0.5) == (test_probs['LSTM'] > 0.5))
print(f'GPT (causal) and LSTM give the same label on {agree:.1%} of test reviews')